# Decomposition Workflow
# $H \to DLA \to e^{i \rho t} \to BDI \to U_{\text{reconstructed}}$

In [35]:
import numpy as np

## Choose model and decomposition parameters
This cell sets the TFIM model size and evolution time.
You can change n, J, h, t and periodic according to your experiment. We remind that we are working with the Hamiltonian:
$
\begin{equation}
    H = -J \sum^{N-2}_{i=0}Z_iZ_{i+1} -h \sum^{N-1}_{i=0}X_i,
\end{equation}
$

In [36]:
# Model parameters
n = 10
J = 1.0
h = 1.0
t = 0.5
rotated = True
periodic = False

print(f"n = {n}, J = {J}, h = {h}, t = {t}, rotated = {rotated}, periodic = {periodic}")

n = 10, J = 1.0, h = 1.0, t = 0.5, rotated = True, periodic = False


In [37]:
from build_TFIM import TFIM_Ham

if n <= 10:
    hamiltonian = TFIM_Ham(n)
    print(f"Hamiltonian shape: {hamiltonian.shape}")
    print(hamiltonian)

Hamiltonian shape: (1024, 1024)
[[-10.   0.   0. ...   0.   0.   0.]
 [  0.  -8.  -1. ...   0.   0.   0.]
 [  0.  -1.  -8. ...   0.   0.   0.]
 ...
 [  0.   0.   0. ...   8.  -1.   0.]
 [  0.   0.   0. ...  -1.   8.   0.]
 [  0.   0.   0. ...   0.   0.  10.]]


## Build DLA generators (Pauli-word form)
We generate TFIM Pauli words and grow the DLA closure.

In [38]:
from find_DLA import tfim_pauliwords_gen, dla_pauli_words

generators = tfim_pauliwords_gen(n, rotated=rotated, periodic=periodic)
dla_words = dla_pauli_words(generators)

print("Initial generators:", len(generators))
print("DLA size:", len(dla_words))
print("First few DLA words:", dla_words[:3])

Initial generators: 19
DLA size: 190
First few DLA words: [('X', 'X', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), ('I', 'X', 'X', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), ('I', 'I', 'X', 'X', 'I', 'I', 'I', 'I', 'I', 'I')]


## Map to the Majorana/isomorphic so(2n) matrix
This creates the isomorphism $\rho(iH)$ used before exponentiation.

In [39]:
from build_isomorphism import map_to_majarana, build_so_matrix

maj_mapping = map_to_majarana(generators, J=J, h=h)
print(maj_mapping)
rho_iH = build_so_matrix(maj_mapping, n)

print("rho(iH) shape:", rho_iH.shape)
print("rho(iH) skew Hermitian?", np.allclose(rho_iH.T, -rho_iH, atol=1e-10))
print(rho_iH)

{('X', 'X', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'): (1.0, (0, 3)), ('I', 'X', 'X', 'I', 'I', 'I', 'I', 'I', 'I', 'I'): (1.0, (2, 5)), ('I', 'I', 'X', 'X', 'I', 'I', 'I', 'I', 'I', 'I'): (1.0, (4, 7)), ('I', 'I', 'I', 'X', 'X', 'I', 'I', 'I', 'I', 'I'): (1.0, (6, 9)), ('I', 'I', 'I', 'I', 'X', 'X', 'I', 'I', 'I', 'I'): (1.0, (8, 11)), ('I', 'I', 'I', 'I', 'I', 'X', 'X', 'I', 'I', 'I'): (1.0, (10, 13)), ('I', 'I', 'I', 'I', 'I', 'I', 'X', 'X', 'I', 'I'): (1.0, (12, 15)), ('I', 'I', 'I', 'I', 'I', 'I', 'I', 'X', 'X', 'I'): (1.0, (14, 17)), ('I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'X', 'X'): (1.0, (16, 19)), ('Z', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'): (-1.0, (0, 1)), ('I', 'Z', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'): (-1.0, (2, 3)), ('I', 'I', 'Z', 'I', 'I', 'I', 'I', 'I', 'I', 'I'): (-1.0, (4, 5)), ('I', 'I', 'I', 'Z', 'I', 'I', 'I', 'I', 'I', 'I'): (-1.0, (6, 7)), ('I', 'I', 'I', 'I', 'Z', 'I', 'I', 'I', 'I', 'I'): (-1.0, (8, 9)), ('I', 'I', 'I', 'I', 'I', 'Z', 'I', 'I', 'I', '

## Exponentiate to get the matrix to decompose
Now build $U(t)=\exp(t\,\rho(iH))$ and verify unitarity/orthogonality.

In [40]:
from BDI_decomp import from_generator

U_t = from_generator(rho_iH, t=t)

print("U(t) shape:", U_t.shape)
print("Is U(t) unitary?", np.allclose(U_t @ U_t.conj().T, np.eye(U_t.shape[0]), atol=1e-10))
print("Is U(t) real (orthogonal case)?", np.allclose(U_t.imag, 0.0, atol=1e-10))

U(t) shape: (20, 20)
Is U(t) unitary? True
Is U(t) real (orthogonal case)? True


## One-step BDI KAK decomposition
Decompose $U(t)$ one step, rebuild $K_1 A K_2$, and check reconstruction error to verify BDI.

In [41]:
from BDI_decomp import bdi, build_kak

k11, k12, theta, k21, k22 = bdi(U_t)
K1, A, K2 = build_kak(k11, k12, theta, k21, k22)
U_rec = K1 @ A @ K2

print("K1 shape:", K1.shape, "A shape:", A.shape, "K2 shape:", K2.shape)
print("One-step reconstruction allclose:", np.allclose(U_t, U_rec, atol=1e-10))
print("Max reconstruction error:", np.max(np.abs(U_t - U_rec)))

K1 shape: (20, 20) A shape: (20, 20) K2 shape: (20, 20)
One-step reconstruction allclose: True
Max reconstruction error: 9.816641933235446e-16


## Recursive BDI
Run the recursive factorization and print operation counts by type.

In [42]:
from BDI_decomp import recursive_bdi

ops = recursive_bdi(U_t, num_iter=None, return_all=False) # This cell needs return_all=False to get the final list of ops
print("Total recursive ops:", len(ops))

types = {}
for _, _, _, op_type in ops:
    types[op_type] = types.get(op_type, 0) + 1

print("Operation type counts:")
for k in sorted(types, key=lambda x: str(x)):
    print(f" {k}: {types[k]}")

print(ops)

"""
Check wether we need to transpose the k's, where is sin and cos on diagonal or off diagonal...
"""

Total recursive ops: 213
Operation type counts:
 a: 52
 a0: 1
 k1: 80
 k2: 80
[(array([[ 0.1729194 ,  0.98493598],
       [-0.98493598,  0.1729194 ]]), 0, 2, 'k1'), (array([[1.]]), 2, 3, 'k1'), (array([[ 9.99999963e-01,  2.71382169e-04],
       [-2.71382169e-04,  9.99999963e-01]]), 3, 5, 'k1'), (array([1.56388272]), 2, 5, 'a'), (array([[1.]]), 2, 3, 'k2'), (array([[-0.08825292, -0.9960981 ],
       [ 0.9960981 , -0.08825292]]), 3, 5, 'k2'), (array([3.14298635, 0.27232512]), 0, 5, 'a'), (array([[-0.99144294, -0.13054077],
       [ 0.13054077, -0.99144294]]), 0, 2, 'k2'), (array([[1.]]), 2, 3, 'k1'), (array([[ 0.99123595,  0.1321033 ],
       [-0.1321033 ,  0.99123595]]), 3, 5, 'k1'), (array([1.57295527]), 2, 5, 'a'), (array([[1.]]), 2, 3, 'k2'), (array([[ 0.99160849, -0.12927721],
       [ 0.12927721,  0.99160849]]), 3, 5, 'k2'), (array([[-1.79129294e-04, -9.99999984e-01],
       [ 9.99999984e-01, -1.79129294e-04]]), 5, 7, 'k1'), (array([[1.]]), 7, 8, 'k1'), (array([[ 0.99819841, -0.059

"\nCheck wether we need to transpose the k's, where is sin and cos on diagonal or off diagonal...\n"

In [43]:
from BDI_verification import verify_bdi_decomposition
verify_bdi_decomposition(U_t, ops)

Max reconstruction error: 6.1035635638521935e-15
Recursive allclose: True


In [44]:
# Print out the determinants of the K1, A, K2 to check if they are in SO(2n) or O(2n)
det_1 = 0
det_m1 = 0
for element in ops:
    if element[3] in ['k1', 'k2']:
        det = np.linalg.det(element[0])
        #print(f"{element[3]} determinant: {det:.4f}")
        if det > 0.5:
            det_1 += 1
        else:
            det_m1 += 1
print(f"Number of K1/K2 with determinant +1: {det_1}")
print(f"Number of K1/K2 with determinant -1: {det_m1}")

Number of K1/K2 with determinant +1: 160
Number of K1/K2 with determinant -1: 0


## Map Back to Pauli Rotations

In [45]:
from map_back import pw_to_majorana, build_majorana_dla_map, map_ops_to_pauli


In [46]:
mapping = build_majorana_dla_map(dla_words)
print(f"Majorana DLA map size: {len(mapping)}")


Majorana DLA map size: 190


In [47]:
pauli_decomp = map_ops_to_pauli(ops, mapping, time=t)
print(f"Pauli decomposition: {len(pauli_decomp)} gates")
print(pauli_decomp)


Pauli decomposition: 190 gates
[(('Z', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), 0.6985016862881988, 'k1'), (('I', 'Y', 'Y', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), 0.00013569108612239303, 'k1'), (('I', 'X', 'Y', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), 0.7819413604034551, 'a'), (('I', 'Y', 'Y', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), -0.8295821067420492, 'k2'), (('X', 'X', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), 1.5714931725852388, 'a'), (('Y', 'Z', 'Y', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), -0.13616255753973683, 'a'), (('Z', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), -1.5053391268452667, 'k2'), (('I', 'Y', 'Y', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), 0.06624528967504655, 'k1'), (('I', 'X', 'Y', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), 0.7864776347826317, 'a'), (('I', 'Y', 'Y', 'I', 'I', 'I', 'I', 'I', 'I', 'I'), -0.06482002099392242, 'k2'), (('I', 'I', 'Y', 'Y', 'I', 'I', 'I', 'I', 'I', 'I'), -0.7854877280447375, 'k1'), (('I', 'I', 'I', 'I', 'Z', 'I', 'I', 'I', 'I', 'I'), -0.030017740157758165, 'k1'

## Verify Reconstruction

In [48]:
from scipy.linalg import expm

if n <= 10:
    # Pauli Matrix Builders
    I2 = np.eye(2, dtype=complex)
    X = np.array([[0, 1], [1, 0]], dtype=complex)
    Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
    Z = np.array([[1, 0], [0, -1]], dtype=complex)

    Pauli_matrices = {"I": I2, "X": X, "Y": Y, "Z": Z}

    # Function to convert a Pauli word to its matrix representation
    def pauli_word_to_matrix(pw):
        mat = Pauli_matrices[pw[0]]
        for p in pw[1:]:
            mat = np.kron(mat, Pauli_matrices[p])
        return mat

    def pauli_rotation(word, coeff):
        P = pauli_word_to_matrix(word)
        return expm(-1.0j * coeff * P)

    U_rec_pauli = np.eye(2**n, dtype=complex)

    for word, coeff, op_type in (pauli_decomp):
        gate = pauli_rotation(word, coeff)
        U_rec_pauli = gate @ U_rec_pauli


In [ ]:
import pennylane as qml

def pauli_word_to_string(pw):
    return "".join(p for p in pw if p != "I")

def pauli_word_to_wires(pw):
    return [i for i, p in enumerate(pw) if p != "I"]

paulirot_decomp = [
    (coeff, pauli_word_to_string(pw), pauli_word_to_wires(pw), typ)
    for pw, coeff, typ in pauli_decomp
]

def kak_time_evolution(time):
    for coeff, pauli_str, wires, typ in paulirot_decomp[::-1]:
        if typ == "a0":
            coeff = coeff * time  # re-scales the divided-out t back in
        qml.PauliRot(2 * coeff, pauli_str, wires=wires)


def phase_align_error(U_ref, U_test):
    # Compare unitaries modulo a physically irrelevant global phase.
    d = U_ref.shape[0]
    phase = np.angle(np.trace(U_test @ U_ref.conj().T) / d)
    U_aligned = np.exp(-1j * phase) * U_test
    return phase, np.linalg.norm(U_ref - U_aligned), np.allclose(U_ref, U_aligned, atol=1e-8)


# Example device / qnode
dev = qml.device("default.qubit", wires=n)

@qml.qnode(dev)
def circuit(time):
    kak_time_evolution(time)
    return qml.state()

if n <= 10:
    U_Hamiltonian = expm(-1.0j * hamiltonian * t)
    U_circ = qml.matrix(circuit)(t)

    raw_close = np.allclose(U_Hamiltonian, U_circ, atol=1e-8)
    raw_norm = np.linalg.norm(U_Hamiltonian - U_circ)
    phase, aligned_norm, aligned_close = phase_align_error(U_Hamiltonian, U_circ)

    print(f"Raw matrix close? {raw_close}")
    print(f"Raw norm difference: {raw_norm}")
    print(f"Global phase offset (rad): {phase}")
    print("\n")
    print(f"Close up to global phase? {aligned_close}")
    print(f"Phase-aligned norm difference: {aligned_norm}")
else:
    print(f"n={n} > 10: full Hilbert space verification skipped.")

print("\nCircuit: ")
qml.draw_mpl(circuit)(t)